In [1]:
import ollama

In [2]:
import os 


In [3]:
import json
import pdfplumber
from crewai import Agent, Task,Crew
from crewai.llm import LLM

In [4]:
print("CWD",os.getcwd())
print("Files in CWD",os.listdir(os.getcwd()))

CWD /Users/apple/Desktop
Files in CWD ['.DS_Store', '.localized', 'Rajesh_Builders_and_Developers_Company_Profile.pdf', 'sentiment.csv', 'rajesh-rag.ipynb', 'company_details.txt']


In [5]:
if os.path.exists("data"):
    with open("data.json", "r") as f:
        data = json.load(f)
else:
    data = {}

In [6]:
from pathlib import Path
from pypdf import PdfReader

pdf_path = Path("Rajesh_Builders_and_Developers_Company_Profile.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

reader = PdfReader(str(pdf_path))

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    print(f"\n--- Page {page_number} ---\n")
    print(text)


--- Page 1 ---

RAJESH BUILDERS & DEVELOPERS | COMPANY PROFILE
Page 1
 RAJESH BUILDERS
 & DEVELOPERS
 COMPANY PROFILE
 Real Estate  Construction  Investments
 Building ideas into lasting spaces
 Tirupati, Andhra Pradesh
 A professional team focused on real estate, construction and property investment solutions.
 Website: www.rajeshbuildersanddevelopers.com
Prepared as a company introduction profile


--- Page 2 ---

RAJESH BUILDERS & DEVELOPERS | COMPANY PROFILE
Page 2
About Us
Rajesh Builders and Developers is presented as a real-estate and construction-focused business based in
Tirupati, Andhra Pradesh. The company profile is centered on helping customers explore property,
construction and investment opportunities with a practical, customer-focused approach. With a team of more
than 15 experienced crew members, the business aims to support projects through coordination, workmanship
and service.
Our Core Services
Real Estate Support — Guidance for customers exploring property optio

In [7]:
llm = LLM(
    model = 'ollama/llama3.2',
    base_url = 'http://localhost:11434',
    temperature = 0
)
print("LLM initialized:", llm)

LLM initialized: <crewai.llm.LLM object at 0x1340a5b50>


In [8]:
pdf_file = os.path.join(os.getcwd(), "Rajesh_Builders_and_Developers_Company_Profile.pdf")
txt_file = os.path.join(os.getcwd(), "company_details.txt")

In [9]:
from pathlib import Path

file_path = Path("company_details.txt")  # or "company_derails.txt"

with file_path.open("r", encoding="utf-8") as file:
    content = file.read()

print(content)

RAJESH BUILDERS & DEVELOPERS - COMPANY PROFILE

Company Overview
----------------
Name: Rajesh Builders & Developers
Location: Tirupati, Andhra Pradesh, India
Core Focus: Premium real estate development
Industry Credentials (as claimed on website):
- ISO 9001:2015 Certified
- RERA Certified
- Experience: 3 years
- Project Portfolio: 500+ projects
- Customer Satisfaction: 98% (claimed)

Contact Information
--------------------
Phone: 9966563335
WhatsApp: +91 99665 63335
Social Media:
- YouTube: @rajeshmanagement26
- Instagram: @rajesh.buildersanddevelopers

Flagship Project: GOLDEN TREE
------------------------------
Status: Under Construction
Type: Premium Residential
Key Features (as claimed):
- RERA Certified
- Smart Home Ready (pre-wired for integration, EV charging, solar readiness)
- Eco-Friendly
- 5-Year Warranty
- Architecture: High ceilings, ample natural light, premium finishes
- Track Record: Claims 100% possession track record with zero project abandonment

Golden Tree Asset

In [10]:
for f in [pdf_file, txt_file]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"File not found: {f}")

In [11]:
print(found_files := [f for f in [pdf_file, txt_file] if os.path.exists(f)])

['/Users/apple/Desktop/Rajesh_Builders_and_Developers_Company_Profile.pdf', '/Users/apple/Desktop/company_details.txt']


In [12]:
print("Files found:", f)

Files found: /Users/apple/Desktop/company_details.txt


In [13]:
def read_pdf(file_path):
    pages=[]
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                print(text)
    return "\n".join(pages)
def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content


In [14]:
knowledge = {
    "pdf_context": read_pdf(pdf_file),
    "text_context": read_txt(txt_file)
}
knowledge = knowledge | data

RAJESH BUILDERS
& DEVELOPERS
COMPANY PROFILE
Real Estate (cid:127) Construction (cid:127) Investments
Building ideas into lasting spaces
Tirupati, Andhra Pradesh
A professional team focused on real estate, construction and property investment solutions.
Website: www.rajeshbuildersanddevelopers.com
Prepared as a company introduction profile
RAJESH BUILDERS & DEVELOPERS | COMPANY PROFILE Page 1
About Us
Rajesh Builders and Developers is presented as a real-estate and construction-focused business based in
Tirupati, Andhra Pradesh. The company profile is centered on helping customers explore property,
construction and investment opportunities with a practical, customer-focused approach. With a team of more
than 15 experienced crew members, the business aims to support projects through coordination, workmanship
and service.
Our Core Services
Real Estate Support — Guidance for customers exploring property options and real-estate opportunities.
Construction — Construction-related support wit

In [15]:
rag_agent = Agent(
    role = "docemnet analyzer",
    goal = "Analyze the document and provide insights",
    backstory = "You are a document analyzer. You will be given a PDF and a text file. Your task is to analyze the content of these documents and provide insights based on the information contained within them.",

llm = llm,
allow_delegation = False,
verbose = True,
)
question = "Where is the company located?"


In [16]:
task = Task(
    description  = f"""
    Document :
    {knowledge}
    question : {question},
    instructions : Analyze the document and provide insights based on the information contained within it.
    if answer is not found in the document, respond with "I don't know".
    if answer is found in the document, provide a detailed response with relevant information and context.
    do not add any information that is not present in the document.
    final answer:
    """,
    expected_output = "Answer to the question based on the document content.",
    agent = rag_agent,
)


In [17]:
crew = Crew(
    agents = [rag_agent],
    tasks = [task],
    llm = llm,
    verbose = True,
)


In [18]:
result = await crew.kickoff_async()
print("Result:", result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3a0052b7-1f7e-4dd6-bd91-8a32590ae283                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Document :                                                                                                 │
│      {'pdf_context': '', 'text_context': "RAJESH BUILDERS & DEVELOPERS - COMPANY PROFILE\n\nCompany             │
│  Overview\n----------------\nName: Rajesh Builders & Developers\nLocation: Tirupati, Andhra Pradesh,            │
│  India\nCore Focus: Premium real estate development\nIndustry Credentials (as claimed on website):\n- ISO       │
│  9001:2015 Certified\n- RERA Certified\n- Experience: 3 years\n- Project Portfolio: 500+ projects\n- Customer   │
│  Satisfaction: 98% (claimed)\n\nContact Information\n--------------------\nPhone: 9966563335\nWhatsApp: +91     │
│  99665 63335\nSocial Media:\n- YouTube: @rajeshmanagement26\n- Instagram:                                       │
│  @rajesh.buildersanddevelopers\n\nFlagship Project: GOLDEN TREE\n------------------------------\nStatus: Under  │
│  Construction\nType: Premium Residential\nKey Features (as claimed):\n- RERA Certified\n- Smart Home Ready      │
│  (pre-wired for integration, EV charging, solar readiness)\n- Eco-Friendly\n- 5-Year Warranty\n- Architecture:  │
│  High ceilings, ample natural light, premium finishes\n- Track Record: Claims 100% possession track record      │
│  with zero project abandonment\n\nGolden Tree Assets (Source-supported):\n- 3D Walkthrough available\n- 4 Site  │
│  Update Videos available\n- 2 Site Map Views available\n\nFuture Projects\n----------------\n- Golden Tree 2    │
│  (Coming Soon)\n- Golden Tree 3 (Coming Soon)\n\nVerification Note\n-----------------\nAll information above    │
│  is extracted directly from the provided website HTML. Company credentials, satisfaction rates, and project     │
│  track records are self-reported claims from the company's official website and have not been independently     │
│  verified.\n"}                                                                                                  │
│      question : Where is the company located?,                                                                  │
│      instructions : Analyze the document and provide insights based on the information contained within it.     │
│      if answer is not found in the document, respond with "I don't know".                                       │
│      if answer is found in the document, provide a detailed response with relevant information and context.     │
│      do not add any information that is not present in the document.                                            │
│      final answer:                                                                                              │
│                                                                                                                 │
│  ID: 8c7c9cd0-9ddf-4626-81bf-b926f02f2dc4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: docemnet analyzer                                                                                       │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Document :                                                                                                 │
│      {'pdf_context': '', 'text_context': "RAJESH BUILDERS & DEVELOPERS - COMPANY PROFILE\n\nCompany             │
│  Overview\n----------------\nName: Rajesh Builders & Developers\nLocation: Tirupati, Andhra Pradesh,            │
│  India\nCore Focus: Premium real estate development\nIndustry Credentials (as claimed on website):\n- ISO       │
│  9001:2015 Certified\n- RERA Certified\n- Experience: 3 years\n- Project Portfolio: 500+ projects\n- Customer   │
│  Satisfaction: 98% (claimed)\n\nContact Information\n--------------------\nPhone: 9966563335\nWhatsApp: +91     │
│  99665 63335\nSocial Media:\n- YouTube: @rajeshmanagement26\n- Instagram:                                       │
│  @rajesh.buildersanddevelopers\n\nFlagship Project: GOLDEN TREE\n------------------------------\nStatus: Under  │
│  Construction\nType: Premium Residential\nKey Features (as claimed):\n- RERA Certified\n- Smart Home Ready      │
│  (pre-wired for integration, EV charging, solar readiness)\n- Eco-Friendly\n- 5-Year Warranty\n- Architecture:  │
│  High ceilings, ample natural light, premium finishes\n- Track Record: Claims 100% possession track record      │
│  with zero project abandonment\n\nGolden Tree Assets (Source-supported):\n- 3D Walkthrough available\n- 4 Site  │
│  Update Videos available\n- 2 Site Map Views available\n\nFuture Projects\n----------------\n- Golden Tree 2    │
│  (Coming Soon)\n- Golden Tree 3 (Coming Soon)\n\nVerification Note\n-----------------\nAll information above    │
│  is extracted directly from the provided website HTML. Company credentials, satisfaction rates, and project     │
│  track records are self-reported claims from the company's official website and have not been independently     │
│  verified.\n"}                                                                                                  │
│      question : Where is the company located?,                                                                  │
│      instructions : Analyze the document and provide insights based on the information contained within it.     │
│      if answer is not found in the document, respond with "I don't know".                                       │
│      if answer is found in the document, provide a detailed response with relevant information and context.     │
│      do not add any information that is not present in the document.                                            │
│      final answer:                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: docemnet analyzer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the document, the company is located in Tirupati, Andhra Pradesh, India.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│      Document :                                                                                                 │
│      {'pdf_context': '', 'text_context': "RAJESH BUILDERS & DEVELOPERS - COMPANY PROFILE\n\nCompany             │
│  Overview\n----------------\nName: Rajesh Builders & Developers\nLocation: Tirupati, Andhra Pradesh,            │
│  India\nCore Focus: Premium real estate development\nIndustry Credentials (as claimed on website):\n- ISO       │
│  9001:2015 Certified\n- RERA Certified\n- Experience: 3 years\n- Project Portfolio: 500+ projects\n- Customer   │
│  Satisfaction: 98% (claimed)\n\nContact Information\n--------------------\nPhone: 9966563335\nWhatsApp: +91     │
│  99665 63335\nSocial Media:\n- YouTube: @rajeshmanagement26\n- Instagram:                                       │
│  @rajesh.buildersanddevelopers\n\nFlagship Project: GOLDEN TREE\n------------------------------\nStatus: Under  │
│  Construction\nType: Premium Residential\nKey Features (as claimed):\n- RERA Certified\n- Smart Home Ready      │
│  (pre-wired for integration, EV charging, solar readiness)\n- Eco-Friendly\n- 5-Year Warranty\n- Architecture:  │
│  High ceilings, ample natural light, premium finishes\n- Track Record: Claims 100% possession track record      │
│  with zero project abandonment\n\nGolden Tree Assets (Source-supported):\n- 3D Walkthrough available\n- 4 Site  │
│  Update Videos available\n- 2 Site Map Views available\n\nFuture Projects\n----------------\n- Golden Tree 2    │
│  (Coming Soon)\n- Golden Tree 3 (Coming Soon)\n\nVerification Note\n-----------------\nAll information above    │
│  is extracted directly from the provided website HTML. Company credentials, satisfaction rates, and project     │
│  track records are self-reported claims from the company's official website and have not been independently     │
│  verified.\n"}                                                                                                  │
│      question : Where is the company located?,                                                                  │
│      instructions : Analyze the document and provide insights based on the information contained within it.     │
│      if answer is not found in the document, respond with "I don't know".                                       │
│      if answer is found in the document, provide a detailed response with relevant information and context.     │
│      do not add any information that is not present in the document.                                            │
│      final answer:                                                                                              │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  docemnet analyzer                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3a0052b7-1f7e-4dd6-bd91-8a32590ae283                                                                           │
│  Final Output: Based on the document, the company is located in Tirupati, Andhra Pradesh, India.                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Result: Based on the document, the company is located in Tirupati, Andhra Pradesh, India.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯